# GSB 5544 — PA 3.2: Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [73]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

The Ames data set (2,930 home sales in Ames, Iowa; tab-separated) is at the URL below. House 0 is the first row. The variables we need for part 1: `Gr Liv Area` (above-ground living area, sq ft), `Bedroom AbvGr`, `Full Bath`, `Half Bath`, and `SalePrice`; part 2 adds `House Style`.

In [74]:
df_ames = pd.read_csv("https://dlsun.github.io/pods/data/AmesHousing.txt", sep="\t")

df_ames.shape

(2930, 82)

In [93]:
df_ames.columns

Index(['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual',
       'Overall Cond', 'Year Built', 'Year Remod/Add', 'Roof Style',
       'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type',
       'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual',
       'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1',
       'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
       'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF',
       '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr',
       'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional',
       'Fireplaces', 'Fireplace Qu', 'Garage Type', 'Garage Yr Blt',
      

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

In [76]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
df_ames["Bathrooms"] = df_ames["Full Bath"] + 0.5 *df_ames["Half Bath"]

house0 = df_ames.loc[0]
show_vars = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "SalePrice", "Neighborhood", "Year Built", "House Style"]

house0[show_vars]


Gr Liv Area        1656
Bedroom AbvGr         3
Bathrooms           1.0
SalePrice        215000
Neighborhood      NAmes
Year Built         1960
House Style      1Story
Name: 0, dtype: object

In [77]:
features = ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]
X = df_ames[features].astype(float)

X_z = (X - X.mean())/ X.std()

diff = X_z - X_z.loc[0]

In [78]:
df_ames["dist_euclid"] = np.sqrt((diff**2).sum(axis=1))
df_ames["dist_euclid"].sort_values().head(10)

0       0.000000
1226    0.009891
1940    0.017804
1357    0.019782
291     0.019782
758     0.019782
2637    0.023738
618     0.023738
2700    0.031651
1541    0.031651
Name: dist_euclid, dtype: float64

In [79]:
df_ames["dist_manhattan"] = diff.abs().sum(axis=1)
df_ames["dist_manhattan"].sort_values().head(10)

0       0.000000
1226    0.009891
1940    0.017804
758     0.019782
1357    0.019782
291     0.019782
618     0.023738
2637    0.023738
2700    0.031651
1541    0.031651
Name: dist_manhattan, dtype: float64

In [80]:
df_cheaper_SalePrice = df_ames[df_ames["SalePrice"]< house0["SalePrice"]]
df_cheaper_SalePrice.sort_values("dist_euclid")[show_vars + ["dist_euclid"]].head(10)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,SalePrice,Neighborhood,Year Built,House Style,dist_euclid
1226,1661,3,1.0,165500,NAmes,1955,SLvl,0.009891
1940,1647,3,1.0,153000,NAmes,1953,1Story,0.017804
1357,1666,3,1.0,161000,OldTown,1925,2Story,0.019782
758,1666,3,1.0,135000,IDOTRR,1927,1.5Fin,0.019782
291,1666,3,1.0,100000,SWISU,1931,1.5Fin,0.019782
2637,1668,3,1.0,135000,OldTown,1948,1.5Fin,0.023738
618,1644,3,1.0,167000,NAmes,1953,1Story,0.023738
2700,1640,3,1.0,131000,Sawyer,1950,1Story,0.031651
1529,1639,3,1.0,115000,SWISU,1936,1.5Fin,0.033629
179,1633,3,1.0,129000,OldTown,1948,1.5Fin,0.045499


In [81]:
df_cheaper_SalePrice.sort_values("dist_manhattan")[show_vars + ["dist_manhattan"]].head(10)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,SalePrice,Neighborhood,Year Built,House Style,dist_manhattan
1226,1661,3,1.0,165500,NAmes,1955,SLvl,0.009891
1940,1647,3,1.0,153000,NAmes,1953,1Story,0.017804
291,1666,3,1.0,100000,SWISU,1931,1.5Fin,0.019782
758,1666,3,1.0,135000,IDOTRR,1927,1.5Fin,0.019782
1357,1666,3,1.0,161000,OldTown,1925,2Story,0.019782
618,1644,3,1.0,167000,NAmes,1953,1Story,0.023738
2637,1668,3,1.0,135000,OldTown,1948,1.5Fin,0.023738
2700,1640,3,1.0,131000,Sawyer,1950,1Story,0.031651
1529,1639,3,1.0,115000,SWISU,1936,1.5Fin,0.033629
179,1633,3,1.0,129000,OldTown,1948,1.5Fin,0.045499


We added sale price after finding the the home with the colosest distance based on the 3 variables "Liv Gr Area", "Bathrooms", "Bedroom AbvGr". Then found all saleprices that were less than the one of house 0 and found which hosues hasd the least distacne form house 0 if less than house 0 saleprice.

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [ ]:
housestyle = pd.get_dummies(df_ames["House Style"], dtype=float)

In [87]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
X = pd.concat([df_ames[features].astype(float), housestyle],
axis = 1)

diff2 = X - X.loc[0]              # each row minus house A's row
diff2

,Gr Liv Area,Bedroom AbvGr,Bathrooms,1.5Fin,1.5Unf,1Story,2.5Fin,2.5Unf,2Story,SFoyer,SLvl
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-760.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-327.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,454.0,0.0,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-27.0,0.0,1.5,0.0,0.0,-1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2925,-653.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,1.0
2926,-754.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2927,-686.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,1.0,0.0
2928,-267.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [88]:
df_ames["dist_euclid"] = np.sqrt((diff2**2).sum(axis=1))
df_ames["dist_euclid"].sort_values().head(10)

0       0.000000
1800    1.000000
1197    1.414214
1927    1.414214
1550    1.500000
1293    2.000000
2638    2.000000
655     2.061553
1493    2.061553
2795    2.236068
Name: dist_euclid, dtype: float64

In [89]:
df_ames["dist_manhattan"] = diff2.abs().sum(axis=1)
df_ames["dist_manhattan"].sort_values().head(10)

0       0.0
1800    1.0
1927    2.0
1197    2.0
1550    2.5
2795    3.0
508     3.5
1493    3.5
655     3.5
1293    4.0
Name: dist_manhattan, dtype: float64

In [91]:
df_cheaper_SalePrice.sort_values("dist_euclid")[
    show_vars + ["dist_euclid"]
].head(10)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,SalePrice,Neighborhood,Year Built,House Style,dist_euclid
1226,1661,3,1.0,165500,NAmes,1955,SLvl,0.009891
1940,1647,3,1.0,153000,NAmes,1953,1Story,0.017804
1357,1666,3,1.0,161000,OldTown,1925,2Story,0.019782
758,1666,3,1.0,135000,IDOTRR,1927,1.5Fin,0.019782
291,1666,3,1.0,100000,SWISU,1931,1.5Fin,0.019782
2637,1668,3,1.0,135000,OldTown,1948,1.5Fin,0.023738
618,1644,3,1.0,167000,NAmes,1953,1Story,0.023738
2700,1640,3,1.0,131000,Sawyer,1950,1Story,0.031651
1529,1639,3,1.0,115000,SWISU,1936,1.5Fin,0.033629
179,1633,3,1.0,129000,OldTown,1948,1.5Fin,0.045499


In [92]:
df_cheaper_SalePrice.sort_values("dist_manhattan")[
    show_vars + ["dist_manhattan"]
].head(10)

,Gr Liv Area,Bedroom AbvGr,Bathrooms,SalePrice,Neighborhood,Year Built,House Style,dist_manhattan
1226,1661,3,1.0,165500,NAmes,1955,SLvl,0.009891
1940,1647,3,1.0,153000,NAmes,1953,1Story,0.017804
291,1666,3,1.0,100000,SWISU,1931,1.5Fin,0.019782
758,1666,3,1.0,135000,IDOTRR,1927,1.5Fin,0.019782
1357,1666,3,1.0,161000,OldTown,1925,2Story,0.019782
618,1644,3,1.0,167000,NAmes,1953,1Story,0.023738
2637,1668,3,1.0,135000,OldTown,1948,1.5Fin,0.023738
2700,1640,3,1.0,131000,Sawyer,1950,1Story,0.031651
1529,1639,3,1.0,115000,SWISU,1936,1.5Fin,0.033629
179,1633,3,1.0,129000,OldTown,1948,1.5Fin,0.045499


I can see some differences between the Euclidean and Manhattan distance results, although the top two closest houses are the same. Adding the dummy variables for House Style did not change the most closely related houses very much, but it did change the order of some of the other similar houses. This suggests that the results are somewhat sensitive to the distance metric and to including House Style, but the closest matches are still fairly consistent.

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [94]:
Zoning = pd.get_dummies(df_ames["MS Zoning"], dtype=float)

In [95]:
Street = pd.get_dummies(df_ames["Street"], dtype=float)

In [99]:
Fence = pd.get_dummies(df_ames["Fence"], dtype=float)

In [114]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
features = ["Fireplaces", "Garage Area", "Lot Area"]
X = pd.concat([df_ames[features].astype(float), Street, Zoning, Fence],
axis = 1)
show_vars = ["Fireplaces", "Garage Area", "Lot Area", "Street", "MS Zoning", "Fence", "SalePrice"]

diff3 = X - X.loc[0]              # each row minus house A's row
diff3

,Fireplaces,Garage Area,Lot Area,Grvl,Pave,A (agr),C (all),FV,I (all),RH,RL,RM,GdPrv,GdWo,MnPrv,MnWw
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-2.0,202.0,-20148.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-1.0,0.0,0.0,0.0,1.0,0.0
2,-2.0,-216.0,-17503.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,-6.0,-20610.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.0,-46.0,-17940.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,-2.0,60.0,-23833.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2926,-2.0,-44.0,-22885.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2927,-2.0,-528.0,-21329.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2928,-1.0,-110.0,-21760.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [115]:
df_ames["dist_euclid"] = np.sqrt((diff3**2).sum(axis=1))
df_ames["dist_euclid"].sort_values().head(10)

0          0.000000
1013     550.989111
2903     580.491171
2229     899.077305
2282    1079.737931
2686    1350.854174
505     1812.027318
2294    2217.680320
2338    2880.336786
834     3072.016927
Name: dist_euclid, dtype: float64

In [116]:
df_ames["dist_manhattan"] = diff3.abs().sum(axis=1)
df_ames["dist_manhattan"].sort_values().head(10)

0          0.0
1013     583.0
2903     782.0
2229     942.0
2686    1401.0
2282    1522.0
505     1873.0
2294    2358.0
2338    2926.0
834     3084.0
Name: dist_manhattan, dtype: float64

In [117]:
df_cheaper_SalePrice = df_ames[
    df_ames["SalePrice"] < house0["SalePrice"]
].copy()

In [118]:
df_cheaper_SalePrice.sort_values("dist_euclid")[
    show_vars + ["dist_euclid"]
].head(10)

,Fireplaces,Garage Area,Lot Area,Street,MS Zoning,Fence,SalePrice,dist_euclid
1013,2,495.0,31220,Pave,RL,NaN,115000,550.989111
2903,0,270.0,31250,Pave,A (agr),NaN,81500,580.491171
2229,2,484.0,32668,Pave,RL,NaN,200624,899.077305
2282,1,1356.0,32463,Pave,RL,NaN,168000,1079.737931
2294,2,672.0,33983,Pave,RL,GdPrv,196000,2217.680320
2338,0,572.0,34650,Pave,RL,NaN,145000,2880.336786
834,0,538.0,28698,Pave,RL,NaN,185000,3072.016927
2273,2,995.0,35133,Grvl,RL,NaN,186700,3395.270240
2298,0,444.0,27697,Pave,RL,NaN,80000,4073.866591
2764,0,390.0,36500,Pave,RL,NaN,190000,4732.013102


In [119]:
df_cheaper_SalePrice.sort_values("dist_manhattan")[
    show_vars + ["dist_manhattan"]
].head(10)

,Fireplaces,Garage Area,Lot Area,Street,MS Zoning,Fence,SalePrice,dist_manhattan
1013,2,495.0,31220,Pave,RL,NaN,115000,583.0
2903,0,270.0,31250,Pave,A (agr),NaN,81500,782.0
2229,2,484.0,32668,Pave,RL,NaN,200624,942.0
2282,1,1356.0,32463,Pave,RL,NaN,168000,1522.0
2294,2,672.0,33983,Pave,RL,GdPrv,196000,2358.0
2338,0,572.0,34650,Pave,RL,NaN,145000,2926.0
834,0,538.0,28698,Pave,RL,NaN,185000,3084.0
2273,2,995.0,35133,Grvl,RL,NaN,186700,3832.0
2298,0,444.0,27697,Pave,RL,NaN,80000,4159.0
2764,0,390.0,36500,Pave,RL,NaN,190000,4870.0


The Euclidean and Manhattan methods gave similar results, although the order of some houses changed. Overall, the closest matches stayed fairly consistent, and scaling helped prevent larger-number variables like lot area from dominating the distance calculation.

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [120]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [125]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc["California Polytechnic State University-San Luis Obispo"]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [129]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
features = ["AdmissionRate", "Undergraduates"]
X = df_college[features].astype(float)

X_z = (X - X.mean())/ X.std()

diff = X_z - X_z.loc[school_name]

In [130]:
df_college["dist_euclid"] = np.sqrt((diff**2).sum(axis=1))
df_college["dist_euclid"].sort_values().head(10)

Institution
California Polytechnic State University-San Luis Obispo    0.000000
University of California-Santa Barbara                     0.309162
DeVry University-Illinois                                  0.593121
University of North Carolina at Chapel Hill                0.596846
Clemson University                                         0.736788
University of Virginia-Main Campus                         0.761296
CUNY Hunter College                                        0.761441
Stony Brook University                                     0.795756
Boston University                                          0.797041
North Carolina State University at Raleigh                 0.826264
Name: dist_euclid, dtype: float64

Based on admission rate and number of undergraduates, UC Santa Barbara is the most similar school to Cal Poly. I determined this using standardized Euclidean distance, where smaller distance values mean the schools are more similar.

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [137]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
Carnegie = pd.get_dummies(df_college["CarnegieClassification"], dtype=float)
Owner = pd.get_dummies(df_college["Ownership"], dtype=float)

In [140]:
X = pd.concat([df_college[features].astype(float), Carnegie, Owner],
axis = 1)

X_z = (X - X.mean()) / X.std()

diff = X_z - X_z.loc[school_name]             # each row minus house A's row
diff

,AdmissionRate,Undergraduates,Associate's Colleges: High Career & Technical-High Nontraditional,Associate's Colleges: High Career & Technical-High Traditional,Associate's Colleges: High Career & Technical-Mixed Traditional/Nontraditional,Associate's Colleges: High Transfer-High Nontraditional,Associate's Colleges: High Transfer-High Traditional,Associate's Colleges: High Transfer-Mixed Traditional/Nontraditional,Baccalaureate Colleges: Arts & Sciences Focus,Baccalaureate Colleges: Diverse Fields,...,Special Focus Four-Year: Other Special Focus Institutions,Special Focus Four-Year: Research Institution,Special Focus Two-Year: Arts & Design,Special Focus Two-Year: Health Professions,Special Focus Two-Year: Other Fields,Special Focus Two-Year: Technical Professions,Tribal Colleges,Private for-profit,Private nonprofit,Public
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,1.747033,-2.058627,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
University of Alabama at Birmingham,2.513736,-1.004855,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
University of Alabama in Huntsville,1.840721,-1.767700,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
Alabama State University,2.941443,-2.264979,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
The University of Alabama,2.077431,1.240300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,-0.095951,-2.684119,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.512607,0.000000,-2.171317
Herzing University-Tampa,2.864953,-2.706132,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,2.046022,-2.171317
Abilene Christian University-Undergraduate Online,3.032415,-2.661463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,2.046022,-2.171317


In [141]:
df_college["dist_manhattan"] = diff.abs().sum(axis=1)
df_college["dist_manhattan"].sort_values().head(10)

Institution
California Polytechnic State University-San Luis Obispo    0.000000
CUNY Hunter College                                        1.072635
CUNY Bernard M Baruch College                              1.516545
CUNY John Jay College of Criminal Justice                  1.586893
CUNY Brooklyn College                                      1.928126
California State Polytechnic University-Pomona             1.985377
CUNY Queens College                                        2.059428
CUNY Lehman College                                        2.346995
Eastern New Mexico University-Main Campus                  2.369223
The University of West Florida                             2.468246
Name: dist_manhattan, dtype: float64

After adding Carnegie classification and ownership, the schools most similar to Cal Poly changed compared with using only admission rate and undergraduate enrollment. The Euclidean and Manhattan methods also produced somewhat different rankings, showing that the results are sensitive to both the variables included and the distance metric used. CUNY Hunter College is closest based on this.

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [142]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

pcip_cols = [col for col in df_college.columns if col.startswith("PCIP")]


In [143]:


X = df_college[pcip_cols].astype(float)

diff = X - X.loc[school_name]

In [144]:

df_college["dist_pcip"] = np.sqrt((diff**2).sum(axis=1))
df_college["dist_pcip"].sort_values().head(10)

Institution
California Polytechnic State University-San Luis Obispo    0.000000
North Carolina State University at Raleigh                 0.084343
Iowa State University                                      0.085054
University of Illinois Urbana-Champaign                    0.111475
Mississippi State University                               0.118799
Texas A & M University-College Station                     0.123973
Clemson University                                         0.128333
Purdue University-Main Campus                              0.134177
Virginia Polytechnic Institute and State University        0.137570
Auburn University                                          0.159167
Name: dist_pcip, dtype: float64

Based on the PCIP field of study proportions, the schools most similar to Cal Poly are North Carolina State University, Iowa State University, University of Illinois, Mississippi State University, and Texas A&M University. I made this decision using Euclidean distance across all PCIP proportion variables, where a smaller distance means the school has a more similar distribution of students across majors.